# Task 2.1 — Feature audit

This notebook audits the HDF5 data product produced by the final C++ extractor. It works on the 7k-row test file or the full ~1.4M-row file.

In [14]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('/work/clas12b/users/skuditha/ALERT/alert_pid/python')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.feature_audit import *

In [15]:
# Set these paths before running.
H5_PATHS = [
    #'/work/clas12b/users/skuditha/ALERT/alert_pid/data/file1.h5',
    '/work/clas12b/users/skuditha/ALERT/alert_pid/data/new_dataset.h5',
]
LABEL_MAP_PATH = '/work/clas12b/users/skuditha/ALERT/alert_pid/config/label_map.json'
OUTDIR = Path('/work/clas12b/users/skuditha/ALERT/alert_pid/reports/feature_audit')

print('Edit H5_PATHS first.')

Edit H5_PATHS first.


In [16]:
# Load dataset
ds = load_audit_dataset(H5_PATHS, LABEL_MAP_PATH)
print({'n_rows': ds.n_rows, 'n_features': ds.n_features, 'files': [str(p) for p in ds.paths]})
print(ds.feature_names)

{'n_rows': 1208319, 'n_features': 38, 'files': ['/work/clas12b/users/skuditha/ALERT/alert_pid/data/new_dataset.h5']}
['px', 'py', 'pz', 'p', 'pt', 'theta', 'phi', 'vx', 'vy', 'vz', 'vr', 'v3', 'n_hits', 'sum_adc', 'path', 'dEdx', 'dedx_recomputed', 'p_drift', 'sum_residuals', 'residual_per_hit', 'adc_per_hit', 'tof_time', 'pathlength', 'cluster_x', 'cluster_y', 'cluster_z', 'cluster_energy', 'n_bar', 'n_wedge', 'beta', 'm2', 'log_p', 'log_pt', 'log_sum_adc', 'log_path', 'log_dEdx', 'log_dedx_recomputed', 'log_cluster_energy']


In [17]:
class_balance = compute_class_balance(ds)
feature_summary = compute_feature_summary(ds)
mask_summary = compute_mask_summary(ds)
unit_sanity = infer_unit_sanity(ds)
pathologies = detect_pathologies(ds)
separation = compute_separation_table(ds)
pair_focus = pair_focus_summary(ds)

class_balance

,class_index,class_name,count,fraction
0,0,proton,191011,0.158080
1,1,deuteron,221182,0.183049
2,2,triton,219291,0.181484
3,3,helium3,300151,0.248404
4,4,helium4,276684,0.228983


In [18]:
feature_summary.head(20)

,feature,valid_count,invalid_count,valid_fraction,raw_min,raw_max,valid_min,valid_max,valid_mean,valid_std,zeros_in_stored_values
0,m2,1188722,19597,0.983782,0.000000,5.147181e+08,0.638005,5.147181e+08,5.261052e+06,9.573733e+06,19597
1,adc_per_hit,1208319,0,1.000000,30.000000,3.822333e+03,30.000000,3.822333e+03,8.103642e+02,6.175537e+02,0
2,beta,1208319,0,1.000000,0.054747,1.199784e+00,0.054747,1.199784e+00,3.936574e-01,1.980406e-01,0
3,cluster_energy,1208319,0,1.000000,0.393653,3.757483e+01,0.393653,3.757483e+01,8.049987e+00,6.282318e+00,0
4,cluster_x,1208319,0,1.000000,-89.876656,8.987666e+01,-89.876656,8.987666e+01,-1.178008e+00,6.288844e+01,0
5,cluster_y,1208319,0,1.000000,-89.876656,8.987666e+01,-89.876656,8.987666e+01,1.981652e-02,6.216999e+01,0
6,cluster_z,1208319,0,1.000000,-187.876007,1.826856e+02,-187.876007,1.826856e+02,-3.301939e+00,7.693170e+01,8
7,dEdx,1208319,0,1.000000,0.384752,3.713100e+02,0.384752,3.713100e+02,8.142985e+01,6.614081e+01,0
8,dedx_recomputed,1208319,0,1.000000,0.384752,3.713101e+02,0.384752,3.713101e+02,8.142985e+01,6.614081e+01,0
9,log_cluster_energy,1208319,0,1.000000,-0.932286,3.626334e+00,-0.932286,3.626334e+00,1.658063e+00,1.049697e+00,0


In [19]:
mask_summary

,metric,value
0,rows_with_any_masked_feature,1.959700e+04
1,rows_with_no_masked_feature,1.188722e+06
2,mean_invalid_features_per_row,1.621840e-02
3,max_invalid_features_in_row,1.000000e+00


In [20]:
unit_sanity

,check,value,comment
0,p_median,584.100037,"Large O(1) suggests GeV/c, O(100-1000) suggest..."
1,tof_time_median_ns,1.107376,Expected ns-scale positive cluster timing.
2,pathlength_median,114.017540,Check whether pathlength looks mm-scale rather...
3,beta_median_stored,0.346772,Should be comfortably below 1 for most rows.
4,beta_median_recomputed_mm_ns,0.346772,Recomputed with c = 299.792458 mm/ns.
5,frac_beta_gt_1p0_stored,0.016218,Diagnostic only; no row cuts in audit.
6,frac_beta_gt_1p0_recomputed,0.016218,High value flags a unit mismatch or timing pat...
7,frac_m2_negative,0.000000,"Negative m2 can occur, but large fractions des..."


In [21]:
pathologies

,pathology,count
0,nonfinite_stored_values,0
1,rows_with_any_nonfinite_stored_value,0
2,valid_time_le_zero,0
3,valid_pathlength_le_zero,0
4,valid_p_le_zero,0
5,valid_dEdx_le_zero,0
6,valid_cluster_energy_le_zero,0
7,valid_beta_le_zero,0
8,valid_beta_gt_1p2,0


In [22]:
separation.head(15)

,feature,fisher_score
0,log_sum_adc,1.934982
1,log_dedx_recomputed,1.865653
2,log_dEdx,1.865653
3,cluster_energy,1.666259
4,adc_per_hit,1.640014
5,log_cluster_energy,1.495595
6,sum_adc,1.487675
7,dedx_recomputed,1.334685
8,dEdx,1.334685
9,beta,0.205857


In [23]:
pair_focus

,feature,class_name,count,median,p16,p84
0,p,deuteron,221182,6.687894e+02,4.241778e+02,1.184587e+03
1,p,helium4,276684,5.496478e+02,3.980743e+02,8.676329e+02
2,tof_time,deuteron,221182,1.017998e+00,6.201248e-01,1.600128e+00
3,tof_time,helium4,276684,1.244247e+00,9.571228e-01,1.625313e+00
4,pathlength,deuteron,221182,1.140175e+02,9.108238e+01,1.330564e+02
5,pathlength,helium4,276684,1.140175e+02,9.108238e+01,1.493106e+02
6,dEdx,deuteron,221182,2.436784e+01,1.124093e+01,6.047642e+01
7,dEdx,helium4,276684,1.373066e+02,9.225360e+01,2.043613e+02
8,cluster_energy,deuteron,221182,2.937158e+00,1.044163e+00,5.733002e+00
9,cluster_energy,helium4,276684,1.435900e+01,9.054011e+00,1.951204e+01


In [24]:
corr = compute_correlation_matrix(ds)
high_corr = high_correlation_pairs(corr, threshold=0.95)
high_corr.head(30)

,feature_a,feature_b,corr
0,dEdx,dedx_recomputed,1.000000
1,log_dEdx,log_dedx_recomputed,1.000000
2,p,p_drift,0.999999
3,sum_residuals,residual_per_hit,0.989295
4,path,log_path,0.984668
5,sum_adc,adc_per_hit,0.982745
6,log_sum_adc,log_dedx_recomputed,0.972093
7,log_sum_adc,log_dEdx,0.972093
8,p_drift,log_p,0.952942
9,p,log_p,0.952654


In [25]:
OUTDIR.mkdir(parents=True, exist_ok=True)
plot_feature_histograms(ds, KEY_PHYSICS_FEATURES, OUTDIR / 'key_histograms')
plot_feature_histograms(ds, DEFAULT_FEATURE_NAMES, OUTDIR / 'histograms')
plot_scatter_by_class(ds, 'p', 'beta', OUTDIR / 'beta_vs_p.png')
plot_scatter_by_class(ds, 'p', 'm2', OUTDIR / 'm2_vs_p.png')
plot_scatter_by_class(ds, 'p', 'dEdx', OUTDIR / 'dEdx_vs_p.png')
plot_scatter_by_class(ds, 'pathlength', 'cluster_energy', OUTDIR / 'cluster_energy_vs_pathlength.png')
plot_correlation_heatmap(corr, OUTDIR / 'correlation_heatmap.png')
print(f'Plots written under {OUTDIR.resolve()}')

Plots written under /ceph24/hallb/clas12/users/skuditha/ALERT/alert_pid/reports/feature_audit


In [26]:
# One-shot batch run
# tables = run_full_feature_audit(H5_PATHS, LABEL_MAP_PATH, OUTDIR)